In [0]:
from pyspark.sql.functions import col
df_customers = spark.read.table("ecommerce.bronze.brz_customers")


In [0]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.column import Column
from pyspark.sql.types import StringType


df_customers = df_customers.withColumn("customer_id", col("customer_id").cast(StringType())) \
        .withColumn("phone", regexp_replace(col("phone"), ".0", ""))

In [0]:

df_customers.filter(col("phone").isNull()).show(5)

In [0]:
from pyspark.sql.column import Column
from typing import Union
from pyspark.sql.functions import concat, when , col, lit , substring

def gen_cust_id(cols: Column, dfs : Column) -> Column:
    return when(cols.isNull(),
        concat(lit("cust"), substring(dfs, 1, 5))) \
    .otherwise(cols)

df_customers = df_customers.withColumn(
    "customer_id", gen_cust_id(col("customer_id"), col("phone"))
)

#### UDF vs Above function 
- remmeber that above function is not UDF
- UDF are written in python & so they are exucuted in Python Engine not Spark engine so they are slower than Above function 
- Always write function in Spark fashion as we wrote above unlesss the functinality is only achivable in Pythin fashion.

In [0]:
df_customers.filter(col("customer_id").isNotNull()).show()

# Repalce all NULL cust id with 'NA'
df_customers = df_customers.withColumn('customer_id', 
    when(col("customer_id").isNull(), lit('NA'))
    .otherwise(col("customer_id"))
)


In [0]:
df_customers.write \
    .format("delta") \
    .option("overwriteSchema", True) \
    .mode("overwrite") \
    .save("s3://sj-dbr-demo-proj/silver_data/slv_customers")

spark.sql("""
          create table if not exists 
          ecommerce.silver.slv_customers
          using delta location 's3://sj-dbr-demo-proj/silver_data/slv_customers' 
          """)

In [0]:
dbutils.notebook.exit("SUCCESS")